# ML HOL - PART 3

## ML Modeling

- In this notebook, we will do a little data exploration before training an XGBoost model with the diamonds dataset using the [Snowpark ML Modeling API](https://docs.snowflake.com/en/developer-guide/snowpark-ml/snowpark-ml-modeling). 
- We also show how to do inference and manage models via Model Registry or as a UDF (See Appendix).

***Note: The Snowpark ML Model API supports `scikit-learn`, `xgboost`, and `lightgbm` models.***

## Pre-installed Packages

This notebook runs on **Container Runtime**, which comes pre-installed with the packages we need:

- `snowflake-ml-python` (includes Snowpark ML modeling, registry, and feature store)
- `seaborn`
- `matplotlib`
- `shap`
- `scikit-learn`

Ensure your container session is **Active** before running the rest of the Notebook.

***Note: The Container Runtime ships with a specific scikit-learn version that cannot be downgraded. The code in this notebook is compatible with scikit-learn 1.3+.***

### Install additional dependencies

The Feature Store API uses `ipywidgets` for interactive display in notebooks. We also pin `xgboost` to match the version available on the warehouse Anaconda channel. Without this, the model trains with a newer xgboost in the container but the predict UDF runs on the warehouse with an older version, causing feature name validation errors.

***Note: This requires an External Access Integration (EAI) for PyPI to be attached to the notebook.***

In [ ]:
!pip install ipywidgets xgboost==3.1.2 -q

### Import Libraries

In [ ]:
# Snowpark for Python
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.version import VERSION
from snowflake.snowpark.functions import udf
import snowflake.snowpark.functions as F
from snowflake.snowpark.types import DecimalType

# Snowpark ML
from snowflake.ml.modeling.xgboost import XGBRegressor
from snowflake.ml.registry import Registry
from snowflake.ml._internal.utils import identifier
import snowflake.ml.modeling.preprocessing as snowml
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.modeling.metrics.correlation import correlation

# OSS ML libs (for distributed HPO on compute pool)
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor as XGBRegressorOSS
from sklearn.metrics import mean_absolute_percentage_error as sklearn_mape
import ray
from ray.util.joblib import register_ray
from snowflake.ml.runtime_cluster import scale_cluster, get_nodes, get_ray_dashboard_url

# data science libs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from snowflake.ml.modeling.metrics import mean_absolute_percentage_error

# misc
import json
import joblib
import cachetools

# warning suppresion
import warnings; warnings.simplefilter('ignore')

### Establish Secure Connection to Snowflake

Notebooks on Container Runtime automatically establish a Snowpark Session when the notebook connects to its compute pool. SQL and Snowpark queries are pushed down to the assigned warehouse. We will use an assigned warehouse, database, and schema as well as the Session object.

In [ ]:
# Get Snowflake Session object
session = get_active_session()

# Set the database and schema context
session.use_database('ML_HOL_DB')
user_suffix = ''.join(filter(str.isdigit, session.get_current_user()))
session.use_schema(f'ML_SCHEMA{user_suffix}')

# Add a query tag to the session.
session.query_tag = {"origin":"sf_sit-is", 
                     "name":"e2e_ml_snowparkpython", 
                     "version":{"major":1, "minor":0,},
                     "attributes":{"is_hol":1, "source":"notebook"}}

# Current Environment Details
print('Connection Established with the following parameters:')
print('User      : {}'.format(session.get_current_user()))
print('Role      : {}'.format(session.get_current_role()))
print('Database  : {}'.format(session.get_current_database()))
print('Schema    : {}'.format(session.get_current_schema()))
print('Warehouse : {}'.format(session.get_current_warehouse()))

### Retrieve values from Feature Store / View

Firstly, we connect to our existing Feature Store.

Then we read the feature values of a registered feature view with `read_feature_view()`.

In [ ]:
from snowflake.ml.feature_store import (
    FeatureStore,
    FeatureView,
    Entity,
    CreationMode,
    FeatureViewStatus,
)

fs = FeatureStore(
    session=session, 
    database=session.get_current_database(), 
    name=session.get_current_schema(), 
    default_warehouse=session.get_current_warehouse(),
    creation_mode=CreationMode.FAIL_IF_NOT_EXIST,
)

fv = fs.get_feature_view('DIAMONDS', '1.0')

diamonds_df = fs.read_feature_view(fv)
diamonds_df

### Feature Transformations

We will illustrate a few of the transformation functions here, but the rest can be found in the [documentation](https://docs.snowflake.com/LIMITEDACCESS/snowflake-ml-preprocessing).

Let's use the `MinMaxScaler` to normalize the `CARAT` column.

In [ ]:
# Normalize the CARAT column
snowml_mms = snowml.MinMaxScaler(input_cols=["CARAT"], output_cols=["CARAT_NORM"])
normalized_diamonds_df = snowml_mms.fit(diamonds_df).transform(diamonds_df)

# Reduce the number of decimals
new_col = normalized_diamonds_df.col("CARAT_NORM").cast(DecimalType(7, 6))
normalized_diamonds_df = normalized_diamonds_df.with_column("CARAT_NORM", new_col)

normalized_diamonds_df

Let's use the `OrdinalEncoder` to transform `COLOR` and `CLARITY` from categorical to numerical values so they are more meaningful.

In [ ]:
# Encode CUT and CLARITY preserve ordinal importance
categories = {
    "CUT": np.array(["IDEAL", "PREMIUM", "VERY_GOOD", "GOOD", "FAIR"]),
    "CLARITY": np.array(["IF", "VVS1", "VVS2", "VS1", "VS2", "SI1", "SI2", "I1", "I2", "I3"]),
}
snowml_oe = snowml.OrdinalEncoder(input_cols=["CUT", "CLARITY"], output_cols=["CUT_OE", "CLARITY_OE"], categories=categories)
ord_encoded_diamonds_df = snowml_oe.fit(normalized_diamonds_df).transform(normalized_diamonds_df)

# Show the encoding
print(snowml_oe._state_pandas)

ord_encoded_diamonds_df

Let's use the `OneHotEncoder` to transform the categorical columns to numerical columns.

This is more for illustration purposes. Using the OrdinalEncoder makes more sense for the diamonds dataset since `CARAT`, `COLOR`, and `CLARITY` all follow a natural ranking order.

In [ ]:
# Encode categoricals to numeric columns
snowml_ohe = snowml.OneHotEncoder(input_cols=["CUT", "COLOR", "CLARITY"], output_cols=["CUT_OHE", "COLOR_OHE", "CLARITY_OHE"])
transformed_diamonds_df = snowml_ohe.fit(ord_encoded_diamonds_df).transform(ord_encoded_diamonds_df)

np.array(transformed_diamonds_df.columns)

transformed_diamonds_df

### Data Exploration

Now that we've transformed our features, let's calculate the correlation using Snowpark ML's `correlation()` function between each pair to better understand their relationships.

*Note: Snowpark ML's pearson correlation function returns a Pandas DataFrame*

In [ ]:
corr_diamonds_df = correlation(df=transformed_diamonds_df)
corr_diamonds_df # This is a Pandas DataFrame

In [ ]:
# # Generate a mask for the upper triangle
mask = np.triu(np.ones_like(corr_diamonds_df, dtype=bool))

# # Create a heatmap with the features
plt.figure(figsize=(7, 4))
plt.rcParams.update({'font.size': 6})
heatmap = sns.heatmap(corr_diamonds_df, mask=mask, cmap="YlGnBu", annot=False, fmt=".2f", vmin=-1, vmax=1, linewidths=.2)

We note that `CARAT` and `PRICE` are highly correlated, which makes sense! Let's take a look at their relationship a bit closer.

*Note: You will have to convert your Snowpark DF to a Pandas DF in order to use matplotlib & seaborn.*

In [ ]:
# Set up a plot to look at CARAT and PRICE
counts = transformed_diamonds_df.to_pandas().groupby(['PRICE', 'CARAT', 'CLARITY_OE']).size().reset_index(name='Count')

fig, ax = plt.subplots(figsize=(20, 20))
plt.title('Price vs Carat', fontsize=20)
ax = sns.scatterplot(data=counts, x='CARAT', y='PRICE', size='Count', hue='CLARITY_OE', markers='o')
ax.grid(axis='y')

# The relationship is not linear - it appears exponential which makes sense given the rarity of the large diamonds
sns.move_legend(ax, "upper left")
sns.despine(left=True, bottom=True)

## ML Model Training and Inference

We we categorize all the features for modeling and build out a full preprocessing `Pipeline`.

This will be useful for both the ML training & inference steps.

In [ ]:
# Categorize all the features for modeling
CATEGORICAL_COLUMNS = ["CUT", "COLOR", "CLARITY"]
CATEGORICAL_COLUMNS_OE = ["CUT_OE", "COLOR_OE", "CLARITY_OE"] # To name the ordinal encoded columns
NUMERICAL_COLUMNS = ["CARAT", "DEPTH", "TABLE_PCT", "X", "Y", "Z"]

categories = {
    "CUT": np.array(["IDEAL", "PREMIUM", "VERY_GOOD", "GOOD", "FAIR"]),
    "CLARITY": np.array(["IF", "VVS1", "VVS2", "VS1", "VS2", "SI1", "SI2", "I1", "I2", "I3"]),
    "COLOR": np.array(['D', 'E', 'F', 'G', 'H', 'I', 'J']),
}

LABEL_COLUMNS = ['PRICE']
OUTPUT_COLUMNS = ['PREDICTED_PRICE']

In [ ]:
# Build the pipeline
preprocessing_pipeline = Pipeline(
    steps=[
            (
                "OE",
                snowml.OrdinalEncoder(
                    input_cols=CATEGORICAL_COLUMNS,
                    output_cols=CATEGORICAL_COLUMNS_OE,
                    categories=categories,
                )
            ),
            (
                "MMS",
                snowml.MinMaxScaler(
                    clip=True,
                    input_cols=NUMERICAL_COLUMNS,
                    output_cols=NUMERICAL_COLUMNS,
                )
            )
    ]
)

diamond_pipe_df = preprocessing_pipeline.fit(diamonds_df).transform(diamonds_df)
diamond_pipe_df


# Other code...
#PIPELINE_FILE = '/tmp/preprocessing_pipeline.joblib'
#joblib.dump(preprocessing_pipeline, PIPELINE_FILE) # We are just pickling it locally first

# You can also save the pickled object into the stage we created earlier for deployment
#session.file.put(PIPELINE_FILE, "@ML_HOL_ASSETS", overwrite=True)

# Load the preprocessing pipeline object from stage- to do this, we download the preprocessing_pipeline.joblib.gz file to the warehouse
# where our notebook is running, and then load it using joblib.
#session.file.get('@ML_HOL_ASSETS/preprocessing_pipeline.joblib.gz', '/tmp')
#PIPELINE_FILE = '/tmp/preprocessing_pipeline.joblib.gz'
#preprocessing_pipeline = joblib.load(PIPELINE_FILE)

### Build a simple XGBoost Regression model

What's happening here? 

- The `model.fit()` function creates a temporary stored procedure that runs on the **warehouse** (not the compute pool). This is a single-node operation. Use a [Snowpark Optimized Warehouse](https://docs.snowflake.com/en/user-guide/warehouses-snowpark-optimized) if you need more memory. We are using an XS Standard Virtual Warehouse here.
- The `model.predict()` function creates a temporary vectorized UDF in the background, which means the input DataFrame is batched as Pandas DataFrames and inference is parallelized across the batches of data.

You can check the query history once you execute the following cell to check.

In [ ]:
# Split the data into train and test sets
diamonds_train_df, diamonds_test_df = diamonds_df.random_split(weights=[0.9, 0.1], seed=0)

# Run the train and test sets through the Pipeline object we defined earlier
train_df = preprocessing_pipeline.fit(diamonds_train_df).transform(diamonds_train_df)
test_df = preprocessing_pipeline.transform(diamonds_test_df)

test_df.write.mode("overwrite").save_as_table("SCORING_DIAMOND_DATA")

In [ ]:
# Define the XGBRegressor
regressor = XGBRegressor(
    input_cols=CATEGORICAL_COLUMNS_OE+NUMERICAL_COLUMNS,
    label_cols=LABEL_COLUMNS,
    output_cols=OUTPUT_COLUMNS
)

# Train
regressor.fit(train_df)

# Predict
result = regressor.predict(test_df)

In [ ]:
# Just to illustrate, we can also pass in a Pandas DataFrame to Snowpark ML's model.predict()
regressor.predict(test_df[CATEGORICAL_COLUMNS_OE+NUMERICAL_COLUMNS].to_pandas())

Let's analyze the results using Snowpark ML's MAPE.

In [ ]:
mape = mean_absolute_percentage_error(df=result, 
                                        y_true_col_names="PRICE", 
                                        y_pred_col_names="PREDICTED_PRICE")

result.select("PRICE", "PREDICTED_PRICE")

In [ ]:
print(f"Mean absolute percentage error: {mape}")

In [ ]:
# Plot actual vs predicted 
g = sns.relplot(data=result["PRICE", "PREDICTED_PRICE"].to_pandas().astype("float64"), x="PRICE", y="PREDICTED_PRICE", kind="scatter")
g.ax.axline((0,0), slope=1, color="r")

plt.show()

### Now, let's use **Distributed** `GridSearchCV()` on the compute pool to find optimal model parameters

We will scale out the compute pool cluster using `scale_cluster()` and distribute the hyperparameter search across worker nodes using Ray as the joblib backend. This runs the grid search directly on the compute pool instead of the warehouse.

*Note: We use OSS scikit-learn's `GridSearchCV` with OSS `xgboost.XGBRegressor` here, distributed via Ray's joblib backend across the scaled compute pool cluster.*

In [ ]:
# Connect to the pre-existing Ray cluster in the Snowflake container runtime
ray.init(address='auto', ignore_reinit_error=True)

# Scale out: expected_cluster_size=3 gives 1 head node + 2 worker nodes
scale_cluster(expected_cluster_size=3)
print(f'Cluster scaled to {len(get_nodes())} nodes')

# Register Ray as the joblib backend so sklearn's n_jobs=-1 distributes across workers
register_ray()

# Print the Ray Dashboard URL for monitoring (open in a new browser tab)
dashboard_url = get_ray_dashboard_url()
print(f'Ray Dashboard: {dashboard_url}')

In [ ]:
# Pull train/test data into pandas for in-memory HPO on compute pool
train_pd = train_df.to_pandas()
test_pd = test_df.to_pandas()

X_train = train_pd[CATEGORICAL_COLUMNS_OE + NUMERICAL_COLUMNS]
y_train = train_pd[LABEL_COLUMNS[0]]
X_test = test_pd[CATEGORICAL_COLUMNS_OE + NUMERICAL_COLUMNS]
y_test = test_pd[LABEL_COLUMNS[0]]

# Define OSS GridSearchCV with XGBoost
grid_search = GridSearchCV(
    estimator=XGBRegressorOSS(),
    param_grid={
        "n_estimators": [100, 200, 300, 400, 500],
        "learning_rate": [0.1, 0.2, 0.3, 0.4, 0.5],
    },
    n_jobs=-1,
    scoring="neg_mean_absolute_percentage_error",
    cv=5
)

# Train — distributed across compute pool workers via Ray
with joblib.parallel_backend('ray'):
    grid_search.fit(X_train, y_train)

In [ ]:
# Scale the cluster back down to free up compute pool resources
scale_cluster(expected_cluster_size=1)

We can access the best estimator directly from the `GridSearchCV` object, which gives us access to all its attributes.

We see that the best estimator has the following parameters: `n_estimators=500` & `learning_rate=0.4`.

In [ ]:
print(grid_search.best_estimator_)

We can also analyze the grid search results.

In [ ]:
# Analyze grid search results
gs_results = grid_search.cv_results_
n_estimators_val = []
learning_rate_val = []
for param_dict in gs_results["params"]:
    n_estimators_val.append(param_dict["n_estimators"])
    learning_rate_val.append(param_dict["learning_rate"])
mape_val = gs_results["mean_test_score"]*-1

gs_results_df = pd.DataFrame(data={
    "n_estimators":n_estimators_val,
    "learning_rate":learning_rate_val,
    "mape":mape_val})

sns.relplot(data=gs_results_df, x="learning_rate", y="mape", hue="n_estimators", kind="line")

plt.show()

This is consistent with the `learning_rate=0.4` and `n_estimator=500` chosen as the best estimator with the lowest MAPE.

Now, let's predict and analyze the results from using the best estimator.

In [ ]:
# Predict using the best estimator from GridSearchCV
y_pred = grid_search.predict(X_test)

# Build a results DataFrame for analysis
result_pd = test_pd.copy()
result_pd['PREDICTED_PRICE'] = y_pred

# Analyze results
optimal_mape = sklearn_mape(result_pd['PRICE'], result_pd['PREDICTED_PRICE'])

print(result_pd[['PRICE', 'PREDICTED_PRICE']].head(10))
print(f"Mean absolute percentage error: {optimal_mape}")

In [ ]:
# Plot actual vs predicted 
g = sns.relplot(data=result_pd[["PRICE", "PREDICTED_PRICE"]].astype("float64"), x="PRICE", y="PREDICTED_PRICE", kind="scatter")
g.ax.axline((0,0), slope=1, color="r")

plt.show()

Let's save our optimal model and its metadata:


In [ ]:
optimal_model = grid_search.best_estimator_
optimal_n_estimators = grid_search.best_estimator_.n_estimators
optimal_learning_rate = grid_search.best_estimator_.learning_rate

val_mape = gs_results_df.loc[(gs_results_df['n_estimators']==optimal_n_estimators) &
                                 (gs_results_df['learning_rate']==optimal_learning_rate), 'mape'].values[0]

### Manage models using Model Registry

Now, with Snowpark ML's [Model Registry](https://docs.snowflake.com/en/developer-guide/snowpark-ml/snowpark-ml-mlops-model-registry), we have a Snowflake native model versioning and deployment framework. This allows us to log models, tag parameters and metrics, track metadata, create versions, and ultimately execute batch inference tasks in a Snowflake warehouse or deploy to a Snowpark Container Service.

First, we will log our models.

Refer to [this Medium post](https://medium.com/snowflake/whats-in-a-name-model-naming-versioning-in-snowpark-model-registry-b5f7105fd6f6) on best practices for model naming & versioning conventions.

In [ ]:
# Get sample input data to pass into the registry logging function
X = train_df.select(CATEGORICAL_COLUMNS_OE+NUMERICAL_COLUMNS).limit(100)

db = identifier._get_unescaped_name(session.get_current_database())
schema = identifier._get_unescaped_name(session.get_current_schema())

# Define model name
model_name = "DIAMONDS_PRICE_PREDICTION"

# Create a registry and log the model
native_registry = Registry(session=session, database_name=db, schema_name=schema)

# Let's first log the very first model we trained
model_ver = native_registry.log_model(
    model_name=model_name,
    version_name='V0',
    model=regressor,
    sample_input_data=X, # to provide the feature schema
    options={"enable_explainability": True}
)

# Add evaluation metric
model_ver.set_metric(metric_name="mean_abs_pct_err", value=mape)

# Add a description
model_ver.comment = "This is the first iteration of our Diamonds Price Prediction model. It is used for demo purposes."

# Now, let's log the optimal model from GridSearchCV
model_ver2 = native_registry.log_model(
    model_name=model_name,
    version_name='V1',
    model=optimal_model,
    sample_input_data=X, # to provide the feature schema
    options={"enable_explainability": True}
)

# Add evaluation metric
model_ver2.set_metric(metric_name="mean_abs_pct_err", value=optimal_mape)

# Add a description
model_ver2.comment = "This is the second iteration of our Diamonds Price Prediction model \
                        where we performed hyperparameter optimization."

In [ ]:
# Let's confirm they were added
native_registry.get_model(model_name).show_versions()

We can see what the default model is when we have multiple versions with the same model name:


In [ ]:
native_registry.get_model(model_name).default.version_name

Now we can use the optimal model to perform inference.

In [ ]:
model_ver = native_registry.get_model(model_name).version('v1')
result_sdf2 = model_ver.run(test_df, function_name="predict")
result_sdf2.show()

In [ ]:
#Save results off to a table.
result_sdf2.write.mode('overwrite').save_as_table('DIAMONDS_SCORED')

### Let's create a stored proc to run inference

In [ ]:
-- Update your Schema Name!

create or replace procedure SCORE_MODEL_FROM_REGISTRY_HOL(model_name string, version_name string, input_table string, scored_table string)
    returns Table()
    language python
    runtime_version = 3.11
    packages =(
        'snowflake-ml-python',
        'snowflake-snowpark-python'
    )
    handler = 'score_model'
    as 
$$    
import snowflake.snowpark as snowpark
from snowflake.snowpark.functions import col
from snowflake.ml.registry import registry

def score_model(session, model_name, version_name, input_table, scored_table): 
    input_data = session.table(input_table)

    #Get model registry
    reg = registry.Registry(session=session, database_name='ML_HOL_DB', schema_name=session.get_current_schema())
    #Get model
    mv = reg.get_model(model_name).version(version_name)
   
    #Score model
    scored_data = mv.run(input_data, function_name='PREDICT')
    #write scored data to snowflake table
    scored_data.write.mode("overwrite").save_as_table(scored_table)
    
    return scored_data
$$;


### The Stored Proc can then be scheduled for batch inference from your orchestration tool or Snowflake Tasks

***Note: Snowflake DAGs: https://docs.snowflake.com/en/user-guide/tasks-graphs#create-a-task-graph***

#### Real-Time Inference via REST API

This workshop uses **batch inference** (scoring a table of data via a stored procedure). For use cases that require **low-latency, real-time predictions** — such as serving a web or mobile application — Snowflake also supports deploying any model from the Model Registry as a managed REST API endpoint on Snowpark Container Services. This gives you a publicly accessible HTTP endpoint with autoscaling, traffic splitting, and observability built in.

```python
# Example: Deploy a registered model as a real-time inference service
model_ver = native_registry.get_model('DIAMONDS_PRICE_PREDICTION').version('V1')
model_ver.create_service(
    service_name='diamonds_price_service',
    service_compute_pool='ML_TEAM_CPU_XS',
    ingress_enabled=True,   # Creates a public HTTP endpoint
)
```

***For details, see: https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/real-time-inference-rest-api***

In [ ]:
CALL SCORE_MODEL_FROM_REGISTRY_HOL('DIAMONDS_PRICE_PREDICTION', 'V0', 'SCORING_DIAMOND_DATA', 'DIAMONDS_SCORED_BY_PROC');

### Model Observability

Once a model is deployed and running inference in production, you'll want to monitor its behavior over time. Snowflake provides native **Model Monitors** that track prediction drift and performance metrics without any external tooling.

With `CREATE MODEL MONITOR`, you can:
- **Track prediction drift** (PSI, KL Divergence) against a baseline dataset to detect when input distributions shift
- **Monitor performance metrics** (MAE, RMSE, MAPE for regression; Accuracy, Precision, Recall, F1 for classification) when ground truth labels become available
- **Segment monitoring** by categorical columns (e.g., monitor drift per diamond `CUT` or `CLARITY` category)
- **Automated refresh** on a configurable schedule, with results surfaced in the Snowsight Models UI

```python
# Example: Create a model monitor for the diamonds price prediction model
# (assumes a source table with predictions and a TIMESTAMP_NTZ column)
#
# CREATE MODEL MONITOR diamonds_monitor WITH
#     MODEL = DIAMONDS_PRICE_PREDICTION
#     VERSION = 'V1'
#     FUNCTION = 'predict'
#     SOURCE = ML_HOL_DB.ML_SCHEMA0.DIAMONDS_SCORED
#     WAREHOUSE = WH0
#     REFRESH_INTERVAL = '1 day'
#     AGGREGATION_WINDOW = '1 day'
#     TIMESTAMP_COLUMN = SCORED_AT
#     PREDICTION_SCORE_COLUMNS = ('PREDICTED_PRICE')
#     BASELINE = ML_HOL_DB.ML_SCHEMA0.SCORING_DIAMOND_DATA;
```

***For details, see: https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/model-observability***

### Model Explainability

Another thing we may want to look at to better understand the predictions are explanations on what the model considers most impactful when generating the predictions. To generate these explanations, we'll use the [built-in explainability function](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/model-explainability) from Snowpark ML. 

Under the hood, this function is based on [Shapley values](https://towardsdatascience.com/the-shapley-value-for-ml-models-f1100bff78d1). During the training process, machine learning models infer relationships between inputs and outputs, and Shapley values are a way to attribute the output of a machine learning model to its input features. By considering all possible combinations of features, Shapley values measure the average marginal contribution of each feature to the model’s prediction. While computationally intensive, the insights gained from Shapley values are invaluable for model interpretability and debugging.

Let's calculate these explanations based on our optimal model now.

In [ ]:
mv_explanations = model_ver.run(test_df, function_name="explain")
mv_explanations

Let's visualize these explanations since it's a bit hard to just interpret the values themselves.

In [ ]:
import shap

mv_explanations_pd = mv_explanations.to_pandas()

mv_explanations_pd_2 = mv_explanations_pd[['CUT_OE_explanation',
                                           'COLOR_OE_explanation',
                                           'CLARITY_OE_explanation',
                                           'CARAT_explanation',
                                           'DEPTH_explanation',
                                           'TABLE_PCT_explanation',
                                           'X_explanation',
                                           'Y_explanation',
                                           'Z_explanation']]

# Wrapping the explanations DataFrame into a SHAP recognized object
shap_exp = shap._explanation.Explanation(mv_explanations_pd_2.values, 
                                         feature_names = mv_explanations_pd_2.columns)
shap.plots.bar(shap_exp)

In [ ]:
plt.figure(figsize=[6,4])


sns.regplot(x= mv_explanations_pd.CARAT, y=mv_explanations_pd.CARAT_explanation, 
            scatter_kws={'s': 75, 'color': 'blue', 'marker': 'o', 'edgecolor': 'white',},  
            line_kws={'color': 'red'})

# Customize the plot
plt.title('Influence of CARAT')
plt.xlabel('CARAT value')
plt.ylabel('CARAT influence')

We see that `CARAT` has the biggest impact on the prediction values (`PRICE`) followed by the `Y dimension`, `CLARITY`, and `COLOR`. This is what we observed in the data exploration phase in the previous notebook too when plotting `PRICE vs CARAT`.

Let's do some clean up now.

In [ ]:
# Clean up
#native_registry.delete_model(model_name)

In [ ]:
# Confirm it was deleted
#native_registry.show_models()

## Conclusion
Congratulations, you have successfully completed this quickstart! Through this quickstart, we were able to showcase Snowpark for Machine Learning through the introduction of Snowpark ML, the Python library and underlying infrastructure for data science and machine learning tasks. Now, you can run data preprocessing, feature engineering, model training, and batch inference in a few lines of code without having to define and deploy stored procedures that package scikit-learn, xgboost, or lightgbm code.

For more information, check out the resources below:

### Related Resources
- [Snowpark ML API Docs](https://docs.snowflake.com/en/developer-guide/snowpark-ml/index)
- [Getting Started with Data Engineering and ML Using Snowpark](https://quickstarts.snowflake.com/guide/getting_started_with_dataengineering_ml_using_snowpark_python/index.html?index=..%2F..index#0)
- [Advanced: Snowpark for Python Data Engineering Guide](https://quickstarts.snowflake.com/guide/data_engineering_pipelines_with_snowpark_python/index.html)
- [Advanced: Snowpark for Python Machine Learning Guide](https://quickstarts.snowflake.com/guide/getting_started_snowpark_machine_learning/index.html)
- [Snowpark for Python Demos](https://github.com/Snowflake-Labs/snowpark-python-demos/blob/main/README.md)
- [Snowpark for Python Developer Docs](https://docs.snowflake.com/en/developer-guide/snowpark/python/index.html)
